In [ ]:
import pyaudio
import numpy as np
import math
import time
from faster_whisper import WhisperModel

def rms(frame):
    """Calculates the volume (Root Mean Square) of an audio chunk."""
    count = len(frame) / 2
    format = "%dh" % (count)
    shorts = np.frombuffer(frame, dtype=np.int16)
    sum_squares = np.sum(shorts.astype(float) ** 2)
    return math.sqrt(sum_squares / count) if count > 0 else 0

def test_local_stt():
    # Audio settings
    FORMAT = pyaudio.paInt16
    CHANNELS = 1
    RATE = 16000
    CHUNK = 1024
    VOLUME_THRESHOLD = 50  # Adjust this if your mic is too quiet/loud
    SILENCE_LIMIT = 1.0    # Seconds of silence before stopping recording

    print("Downloading/Loading Local Whisper Model (base.en)...")
    # Using CPU and int8 to ensure it runs on any machine without complex GPU setup
    model = WhisperModel("base.en", device="cpu", compute_type="int8")
    
    audio = pyaudio.PyAudio()
    stream = None

    try:
        stream = audio.open(format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK)
        print("\n🎙️ RUA's STT Engine is Online (100% Local).")
        print("Speak into the microphone. It will detect your voice, record, and transcribe.")
        print("(Press Ctrl+C to exit)\n")

        while True:
            audio_data = bytearray()
            recording = False
            silence_start = None

            # 1. Listen for voice activity
            while True:
                chunk = stream.read(CHUNK, exception_on_overflow=False)
                vol = rms(chunk)

                if vol > VOLUME_THRESHOLD:
                    if not recording:
                        print("\n[Listening...] ", end="", flush=True)
                        recording = True
                    audio_data.extend(chunk)
                    silence_start = None  # Reset silence timer
                
                elif recording:
                    audio_data.extend(chunk)
                    if silence_start is None:
                        silence_start = time.time()
                    elif time.time() - silence_start > SILENCE_LIMIT:
                        # User stopped speaking
                        break

            # 2. Process the audio chunk
            if len(audio_data) > 0:
                print("\n[Processing...] ", end="", flush=True)
                
                # Convert raw bytes to Float32 array (required by Whisper)
                audio_np = np.frombuffer(audio_data, dtype=np.int16).astype(np.float32) / 32768.0

                # Run local transcription with word timestamps enabled
                segments, info = model.transcribe(audio_np, beam_size=5, word_timestamps=True)
                
                print("\n🗣️ RUA Heard: ", end="")
                for segment in segments:
                    for word in segment.words:
                        # Print word-by-word to the terminal
                        print(word.word, end="", flush=True)
                        # Simulate the real-time typing effect based on the AI's timing
                        time.sleep(word.end - word.start) 
                print("\n" + "-"*40)

    except KeyboardInterrupt:
        print("\nStopping STT engine.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
    finally:
        if stream is not None:
            stream.close()
        audio.terminate()
        print("Microphone closed.")

# Run the test
test_local_stt()
# it is working but only detects english

In [ ]:
import pyaudio
import numpy as np
import math
import time
from faster_whisper import WhisperModel

def rms(frame):
    """Calculates the volume (Root Mean Square) of an audio chunk."""
    count = len(frame) / 2
    format = "%dh" % (count)
    shorts = np.frombuffer(frame, dtype=np.int16)
    sum_squares = np.sum(shorts.astype(float) ** 2)
    return math.sqrt(sum_squares / count) if count > 0 else 0

def test_local_stt():
    # Audio settings
    FORMAT = pyaudio.paInt16
    CHANNELS = 1
    RATE = 16000
    CHUNK = 1024
    
    # 🔧 FIX 1: Raised threshold to ignore background room noise
    VOLUME_THRESHOLD = 300  
    SILENCE_LIMIT = 1.0    

    print("Downloading/Loading Local Whisper Model (Multilingual Base)...")
    # 🔧 FIX 2: Changed "base.en" to "base" to support Hindi and auto-detection
    model = WhisperModel("base", device="cpu", compute_type="int8")
    
    audio = pyaudio.PyAudio()
    stream = None

    try:
        stream = audio.open(format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK)
        print("\n🎙️ RUA's STT Engine is Online (Multilingual + VAD Filter).")
        print("Speak English or Hindi into the microphone.")
        print("(Press Ctrl+C to exit)\n")

        while True:
            audio_data = bytearray()
            recording = False
            silence_start = None

            # 1. Listen for voice activity
            while True:
                chunk = stream.read(CHUNK, exception_on_overflow=False)
                vol = rms(chunk)

                if vol > VOLUME_THRESHOLD:
                    if not recording:
                        print("\n[Listening...] ", end="", flush=True)
                        recording = True
                    audio_data.extend(chunk)
                    silence_start = None  
                
                elif recording:
                    audio_data.extend(chunk)
                    if silence_start is None:
                        silence_start = time.time()
                    elif time.time() - silence_start > SILENCE_LIMIT:
                        break

            # 2. Process the audio chunk
            if len(audio_data) > 0:
                print("\n[Processing...] ", end="", flush=True)
                
                audio_np = np.frombuffer(audio_data, dtype=np.int16).astype(np.float32) / 32768.0

                # 🔧 FIX 3: Added vad_filter=True to aggressively kill "ghost" transcriptions
                segments, info = model.transcribe(
                    audio_np, 
                    beam_size=5, 
                    word_timestamps=True, 
                    vad_filter=True,
                    vad_parameters=dict(min_silence_duration_ms=500)
                )
                
                # Print the detected language
                print(f"\n🌍 Language Detected: {info.language.upper()} (Confidence: {info.language_probability:.2f})")
                print("🗣️ RUA Heard: ", end="")
                
                segment_count = 0
                for segment in segments:
                    segment_count += 1
                    for word in segment.words:
                        print(word.word, end="", flush=True)
                        time.sleep(word.end - word.start) 
                
                # If the VAD filter killed the audio (because it was just noise), print a notice
                if segment_count == 0:
                    print("[Audio discarded as background noise]")
                    
                print("\n" + "-"*40)

    except KeyboardInterrupt:
        print("\nStopping STT engine.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
    finally:
        if stream is not None:
            stream.close()
        audio.terminate()
        print("Microphone closed.")

# Run the test
test_local_stt()
# multilingual but poor accuracy


In [ ]:
import pyaudio
import numpy as np
import math
import time
from faster_whisper import WhisperModel

def rms(frame):
    count = len(frame) / 2
    format = "%dh" % (count)
    shorts = np.frombuffer(frame, dtype=np.int16)
    sum_squares = np.sum(shorts.astype(float) ** 2)
    return math.sqrt(sum_squares / count) if count > 0 else 0

def test_local_stt():
    FORMAT = pyaudio.paInt16
    CHANNELS = 1
    RATE = 16000
    CHUNK = 1024
    
    VOLUME_THRESHOLD = 300  
    SILENCE_LIMIT = 1.0    

    print("Downloading/Loading Local Whisper Model ('Small' for higher accuracy)...")
    # 🔧 FIX 1: Upgraded from "base" to "small" for better Hindi/Accent recognition
    model = WhisperModel("small", device="cpu", compute_type="int8")
    
    audio = pyaudio.PyAudio()
    stream = None

    try:
        stream = audio.open(format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK)
        print("\n🎙️ RUA's STT Engine is Online (High Accuracy 'Small' Model).")
        print("Speak English or Hindi into the microphone.")
        print("(Press Ctrl+C to exit)\n")

        while True:
            audio_data = bytearray()
            recording = False
            silence_start = None

            while True:
                chunk = stream.read(CHUNK, exception_on_overflow=False)
                vol = rms(chunk)

                if vol > VOLUME_THRESHOLD:
                    if not recording:
                        print("\n[Listening...] ", end="", flush=True)
                        recording = True
                    audio_data.extend(chunk)
                    silence_start = None  
                
                elif recording:
                    audio_data.extend(chunk)
                    if silence_start is None:
                        silence_start = time.time()
                    elif time.time() - silence_start > SILENCE_LIMIT:
                        break

            if len(audio_data) > 0:
                print("\n[Processing...] ", end="", flush=True)
                
                audio_np = np.frombuffer(audio_data, dtype=np.int16).astype(np.float32) / 32768.0

                # 🔧 FIX 2 & 3: Added anti-hallucination flags and a Hinglish prompt
                segments, info = model.transcribe(
                    audio_np, 
                    beam_size=5, 
                    word_timestamps=True, 
                    vad_filter=True,
                    vad_parameters=dict(min_silence_duration_ms=500),
                    condition_on_previous_text=False, # Kills the weird ghost subtitles
                    initial_prompt="Hello Rua. How are you? Kaise ho?" # Primes it for Hindi/English
                )
                
                print(f"\n🌍 Language Detected: {info.language.upper()} (Confidence: {info.language_probability:.2f})")
                print("🗣️ RUA Heard: ", end="")
                
                segment_count = 0
                for segment in segments:
                    segment_count += 1
                    for word in segment.words:
                        print(word.word, end="", flush=True)
                        time.sleep(word.end - word.start) 
                
                if segment_count == 0:
                    print("[Audio discarded as background noise]")
                    
                print("\n" + "-"*40)

    except KeyboardInterrupt:
        print("\nStopping STT engine.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
    finally:
        if stream is not None:
            stream.close()
        audio.terminate()
        print("Microphone closed.")

test_local_stt()

# much better than previous ones

Downloading/Loading Local Whisper Model ('Small' for higher accuracy)...


d:\rua_v2\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yanil\.cache\huggingface\hub\models--Systran--faster-whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)



🎙️ RUA's STT Engine is Online (High Accuracy 'Small' Model).
Speak English or Hindi into the microphone.
(Press Ctrl+C to exit)


[Listening...] 
[Processing...] 
🌍 Language Detected: EN (Confidence: 0.38)
🗣️ RUA Heard: [Audio discarded as background noise]

----------------------------------------

[Listening...] 
[Processing...] 
🌍 Language Detected: EN (Confidence: 0.40)
🗣️ RUA Heard:  Rua, how are you? Kaise ho?
----------------------------------------

[Listening...] 
[Processing...] 
🌍 Language Detected: HI (Confidence: 0.90)
🗣️ RUA Heard:  Rua. How are you?
----------------------------------------

[Listening...] 
[Processing...] 
🌍 Language Detected: EN (Confidence: 0.38)
🗣️ RUA Heard: [Audio discarded as background noise]

----------------------------------------

[Listening...] 
[Processing...] 
🌍 Language Detected: EN (Confidence: 0.38)
🗣️ RUA Heard: [Audio discarded as background noise]

----------------------------------------

[Listening...] 
[Processing...] 
🌍 Language 

In [ ]:
import pyaudio
import numpy as np
import math
import queue
import threading
from faster_whisper import WhisperModel

# Queue to hold audio chunks securely between threads
audio_queue = queue.Queue()
is_recording = True

def rms(frame):
    """Calculates the volume of the audio chunk."""
    count = len(frame) / 2
    format = "%dh" % (count)
    shorts = np.frombuffer(frame, dtype=np.int16)
    sum_squares = np.sum(shorts.astype(float) ** 2)
    return math.sqrt(sum_squares / count) if count > 0 else 0

def record_audio(stream, chunk_size, threshold):
    """BACKGROUND THREAD: Constantly listens and pushes active audio to the queue."""
    global is_recording
    while is_recording:
        try:
            chunk = stream.read(chunk_size, exception_on_overflow=False)
            if rms(chunk) > threshold:
                audio_queue.put(chunk)
            else:
                audio_queue.put(None) # Pushes a 'silence' marker
        except Exception:
            break

def test_instant_stt():
    global is_recording
    FORMAT = pyaudio.paInt16
    CHANNELS = 1
    RATE = 16000
    CHUNK = 1024
    VOLUME_THRESHOLD = 300  
    
    print("Loading Faster-Whisper ('Small' Model)...")
    model = WhisperModel("small", device="cpu", compute_type="int8")
    
    audio = pyaudio.PyAudio()
    stream = audio.open(format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK)
    
    # Start the background recording thread
    record_thread = threading.Thread(target=record_audio, args=(stream, CHUNK, VOLUME_THRESHOLD))
    record_thread.start()
    
    print("\n⚡ RUA Instant STT is Online.")
    print("Speak English or Hindi. Words will print as you speak.")
    print("(Press Ctrl+C to exit)\n")
    print("🗣️ ", end="", flush=True)
    
    try:
        audio_buffer = bytearray()
        silence_counter = 0
        
        while True:
            chunk = audio_queue.get()
            
            if chunk is not None:
                audio_buffer.extend(chunk)
                silence_counter = 0
            else:
                silence_counter += 1
                
            # Trigger processing if we hit ~1 second of audio (32000 bytes) 
            # OR a slight pause (0.3s) with at least some audio in the buffer
            if len(audio_buffer) >= 32000 or (silence_counter > 5 and len(audio_buffer) > 4000):
                
                audio_np = np.frombuffer(audio_buffer, dtype=np.int16).astype(np.float32) / 32768.0
                
                # 🔧 FAST PROCESSING: beam_size=1 and no timestamps
                segments, info = model.transcribe(
                    audio_np, 
                    beam_size=1, 
                    condition_on_previous_text=False,
                    initial_prompt="Hello Rua. How are you? Kaise ho?"
                )
                
                for segment in segments:
                    # Print instantly, right next to the previous word
                    print(segment.text + " ", end="", flush=True)
                
                # Clear buffer for the next instant chunk
                audio_buffer = bytearray() 
                
    except KeyboardInterrupt:
        print("\nStopping...")
    finally:
        is_recording = False
        record_thread.join()
        stream.close()
        audio.terminate()
        print("Microphone closed.")

# Run the test
test_instant_stt()

# better than others but ery slow for our project

Loading Faster-Whisper ('Small' Model)...

⚡ RUA Instant STT is Online.
Speak English or Hindi. Words will print as you speak.
(Press Ctrl+C to exit)

🗣️  Hello Rua.  Hello Rua. My name is...  I am in  Data science.  Hello Rua. My name is Kaise.  I am Kaise Yanniler and I am an...  Data science. 
Stopping...
Microphone closed.


In [5]:
import pyaudio
import json
import sys
from vosk import Model, KaldiRecognizer

def test_true_streaming_stt():
    print("Downloading/Loading Vosk RNN-T Model (Indian English)...")
    print("This will take a moment on the first run.")
    
    # Loads the lightweight streaming model specifically tuned for Indian English
    model = Model(lang="en-in") 
    
    # Initialize the recognizer with a 16kHz sample rate
    rec = KaldiRecognizer(model, 16000)
    
    # Initialize microphone
    p = pyaudio.PyAudio()
    stream = p.open(format=pyaudio.paInt16, channels=1, rate=16000, input=True, frames_per_buffer=4000)
    stream.start_stream()
    
    print("\n⚡ RUA's True Streaming STT is Online (RNN-T Architecture).")
    print("Speak naturally. Watch the terminal update instantly.")
    print("(Press Ctrl+C to exit)\n")
    
    try:
        while True:
            # Read a small, continuous stream of audio
            data = stream.read(4000, exception_on_overflow=False)
            
            if rec.AcceptWaveform(data):
                # The user paused/stopped speaking, lock in the final sentence
                result = json.loads(rec.Result())
                if result['text']:
                    # Clear the partial line and print the final locked text
                    sys.stdout.write('\r' + ' ' * 80 + '\r') 
                    print(f"✅ RUA Heard [FINAL]: {result['text']}\n")
            else:
                # The user is currently speaking, print the syllables/words instantly
                partial = json.loads(rec.PartialResult())
                if partial['partial']:
                    # The '\r' forces the terminal to overwrite the same line instantly
                    sys.stdout.write(f"\r🗣️ [Streaming...]: {partial['partial']}")
                    sys.stdout.flush()

    except KeyboardInterrupt:
        print("\n\nStopping STT engine.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
    finally:
        stream.stop_stream()
        stream.close()
        p.terminate()
        print("Microphone closed.")

# Run the test
test_true_streaming_stt()

Downloading/Loading Vosk RNN-T Model (Indian English)...
This will take a moment on the first run.


vosk-model-small-en-in-0.4.zip: 100%|██████████| 35.8M/35.8M [02:41<00:00, 232kB/s]   



⚡ RUA's True Streaming STT is Online (RNN-T Architecture).
Speak naturally. Watch the terminal update instantly.
(Press Ctrl+C to exit)

✅ RUA Heard [FINAL]: hello                                                      

✅ RUA Heard [FINAL]: hello                                                      

✅ RUA Heard [FINAL]: what are you doing it                                      

✅ RUA Heard [FINAL]: hello how are you                                          

✅ RUA Heard [FINAL]: true                                                       

✅ RUA Heard [FINAL]: brewer                                                     

✅ RUA Heard [FINAL]: hello to our what are you doing                            

✅ RUA Heard [FINAL]: electra                                                    

✅ RUA Heard [FINAL]: an extra                                                   

✅ RUA Heard [FINAL]: actually she is not appropriate                            



Stopping STT engine.
Microphone closed.


In [ ]:
import pyaudio
import json
import sys
import os
from vosk import Model, KaldiRecognizer

def test_true_streaming_stt_large():
    # Provide the exact path to your extracted large model
    MODEL_PATH = "../models/vosk-large" 
    
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Error: Could not find the large model at {MODEL_PATH}")
        print("Please download it and extract it to rua/models/vosk-large")
        return

    print("Loading Massive Vosk RNN-T Model (High Accuracy)...")
    print("This will take a lot of RAM. Please wait.")
    
    # Load the heavy model directly from the folder
    model = Model(model_path=MODEL_PATH) 
    
    rec = KaldiRecognizer(model, 16000)
    
    p = pyaudio.PyAudio()
    stream = p.open(format=pyaudio.paInt16, channels=1, rate=16000, input=True, frames_per_buffer=4000)
    stream.start_stream()
    
    print("\n⚡ RUA's True Streaming STT is Online (Large Model).")
    print("Speak naturally. Watch the terminal update instantly.")
    print("(Press Ctrl+C to exit)\n")
    
    try:
        while True:
            data = stream.read(4000, exception_on_overflow=False)
            
            if rec.AcceptWaveform(data):
                result = json.loads(rec.Result())
                if result['text']:
                    sys.stdout.write('\r' + ' ' * 80 + '\r') 
                    print(f"✅ RUA Heard [FINAL]: {result['text']}\n")
            else:
                partial = json.loads(rec.PartialResult())
                if partial['partial']:
                    sys.stdout.write(f"\r🗣️ [Streaming...]: {partial['partial']}")
                    sys.stdout.flush()

    except KeyboardInterrupt:
        print("\n\nStopping STT engine.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
    finally:
        stream.stop_stream()
        stream.close()
        p.terminate()
        print("Microphone closed.")

# Run the test
test_true_streaming_stt_large()

# we will be using this apporach

Loading Massive Vosk RNN-T Model (High Accuracy)...
This will take a lot of RAM. Please wait.

⚡ RUA's True Streaming STT is Online (Large Model).
Speak naturally. Watch the terminal update instantly.
(Press Ctrl+C to exit)

✅ RUA Heard [FINAL]: hello room                                                 

✅ RUA Heard [FINAL]: hello what are you doing                                   

✅ RUA Heard [FINAL]: how are you                                                

✅ RUA Heard [FINAL]: loading machine                                            

✅ RUA Heard [FINAL]: and teen model high accuracy                               

✅ RUA Heard [FINAL]: this will take a lot of repurchase rate                    

✅ RUA Heard [FINAL]: rewa to streaming online                                   

✅ RUA Heard [FINAL]: true train me as treaty is online speak naturally watch the terminal update instantly



Stopping STT engine.
Microphone closed.


In [7]:
import speech_recognition as sr
import numpy as np
from faster_whisper import WhisperModel
import time

def perfect_bilingual_ear():
    print("🧠 Loading RUA's Bilingual Brain ('small' Whisper model)...")
    # Using 'small' for the perfect balance of speed and Hindi/English accuracy
    model = WhisperModel("small", device="cpu", compute_type="int8")
    
    # Initialize the dynamic microphone recognizer
    recognizer = sr.Recognizer()
    
    # Advanced Settings for snappy responses
    recognizer.energy_threshold = 300 # Baseline volume
    recognizer.dynamic_energy_threshold = True # Auto-adjusts to your room's noise
    recognizer.pause_threshold = 0.6 # The exact seconds of silence before it processes your sentence

    print("\n🎙️ RUA's Perfect Ear is Online.")
    print("You can speak in English, Hindi, or mix them both (Hinglish)!")
    print("(Press Ctrl+C to exit)\n")

    try:
        with sr.Microphone(sample_rate=16000) as source:
            print("[Calibrating to room noise for 1 second... Shhh...]")
            recognizer.adjust_for_ambient_noise(source, duration=1.0)
            print("[Calibration Complete. Ready.]\n")

            while True:
                print("🟢 Listening... ", end="", flush=True)
                
                # Listen until the user pauses for 0.6 seconds
                audio = recognizer.listen(source, timeout=None, phrase_time_limit=10)
                print("\r🟡 Processing... ", end="", flush=True)
                
                # Convert the raw audio to the exact mathematical format Whisper needs
                raw_data = audio.get_raw_data()
                audio_np = np.frombuffer(raw_data, dtype=np.int16).astype(np.float32) / 32768.0
                
                # Send to Whisper with anti-hallucination and Hinglish priming
                start_time = time.time()
                segments, info = model.transcribe(
                    audio_np, 
                    beam_size=2, # Small beam size for speed, but enough for accuracy
                    condition_on_previous_text=False, # Kills hallucinations
                    initial_prompt="Hello Rua. How are you? Kaise ho? Kya chal raha hai?" # Primes for Hinglish
                )
                
                # Extract the text
                transcription = "".join([segment.text for segment in segments]).strip()
                latency = time.time() - start_time
                
                if transcription:
                    # Clear the processing line and print the final result
                    print("\r" + " " * 50 + "\r", end="") 
                    print(f"🌍 [{info.language.upper()}] RUA Heard: {transcription}")
                    print(f"⏱️  Speed: {latency:.2f} seconds\n")
                else:
                    print("\r" + " " * 50 + "\r", end="") 

    except KeyboardInterrupt:
        print("\n\n🛑 Stopping RUA's Ear.")
    except Exception as e:
        print(f"\n❌ Error: {e}")

# Run the ear test
perfect_bilingual_ear()

🧠 Loading RUA's Bilingual Brain ('small' Whisper model)...

🎙️ RUA's Perfect Ear is Online.
You can speak in English, Hindi, or mix them both (Hinglish)!
(Press Ctrl+C to exit)

[Calibrating to room noise for 1 second... Shhh...]
[Calibration Complete. Ready.]

🌍 [HI] RUA Heard: Hello Rua. How are you?        
⏱️  Speed: 2.59 seconds

🌍 [HI] RUA Heard: Hello Rua. How are you? Kaise ho? Kya chal raha hai ho?
⏱️  Speed: 2.69 seconds

🌍 [HI] RUA Heard: Hello Rua. Kaise ho? Ka kya kar raha hai?
⏱️  Speed: 2.63 seconds

🌍 [HI] RUA Heard: Hello Rua. My name is..        
⏱️  Speed: 2.55 seconds

🌍 [EN] RUA Heard: Loading Rua is                 
⏱️  Speed: 2.52 seconds

🟢 Listening... 

🛑 Stopping RUA's Ear.


In [11]:
import speech_recognition as sr
import numpy as np
from faster_whisper import WhisperModel
import time
import os
import traceback

def instant_bilingual_ear():
    optimal_threads = min(4, os.cpu_count() or 4)
    print(f"🧠 Loading RUA's Brain (Safely using {optimal_threads} CPU Cores)...")
    
    model = WhisperModel(
        "small", 
        device="cpu", 
        compute_type="int8", 
        cpu_threads=optimal_threads 
    )
    
    recognizer = sr.Recognizer()
    recognizer.energy_threshold = 300
    recognizer.dynamic_energy_threshold = True 
    
    # 🔧 FIX: non_speaking_duration must be less than or equal to pause_threshold
    recognizer.pause_threshold = 0.35 
    recognizer.non_speaking_duration = 0.3 

    print("\n⚡ RUA's Ultra-Fast Bilingual Ear is Online.")
    print("(Press Ctrl+C to exit)\n")

    try:
        with sr.Microphone(sample_rate=16000) as source:
            print("[Calibrating... Shhh...]")
            recognizer.adjust_for_ambient_noise(source, duration=1.0)
            print("[Ready.]\n")

            while True:
                print("🟢 Listening... ", end="", flush=True)
                audio = recognizer.listen(source, timeout=None, phrase_time_limit=10)
                print("\r🟡 Processing... ", end="", flush=True)
                
                raw_data = audio.get_raw_data()
                audio_np = np.frombuffer(raw_data, dtype=np.int16).astype(np.float32) / 32768.0
                
                start_time = time.time()
                
                segments, info = model.transcribe(
                    audio_np, 
                    beam_size=1, 
                    condition_on_previous_text=False, 
                    initial_prompt="Hello Rua. How are you? Kaise ho?"
                )
                
                transcription = "".join([segment.text for segment in segments]).strip()
                latency = time.time() - start_time
                
                if transcription:
                    print("\r" + " " * 50 + "\r", end="") 
                    print(f"🌍 [{info.language.upper()}] Heard: {transcription}")
                    print(f"⏱️  Speed: {latency:.2f} seconds\n")
                else:
                    print("\r" + " " * 50 + "\r", end="") 

    except KeyboardInterrupt:
        print("\n\n🛑 Stopping RUA's Ear.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        traceback.print_exc()

# Run the ear test
instant_bilingual_ear()

🧠 Loading RUA's Brain (Safely using 4 CPU Cores)...

⚡ RUA's Ultra-Fast Bilingual Ear is Online.
(Press Ctrl+C to exit)

[Calibrating... Shhh...]
[Ready.]

🌍 [EN] Heard: Hello Rua. How are you?            
⏱️  Speed: 2.73 seconds

🟢 Listening... 

🛑 Stopping RUA's Ear.
